# Customer Churn Prediction - End-to-End ML Project

This notebook is a **portfolio-grade churn prediction project** built around the provided `churn.csv` dataset.

## Project goals
- Inspect and clean the dataset
- Perform EDA
- Engineer useful features
- Build preprocessing and modeling pipelines
- Compare multiple ML models
- Tune top models
- Interpret results
- Save the final model for deployment

> **Target column:** `churn_risk_score`  
> Assumption: `1 = churn / high churn risk`, `0 = not churn`

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
import joblib

pd.set_option("display.max_columns", None)
sns.set(style="whitegrid")

## 2. Load the dataset

In [ ]:
df = pd.read_csv("churn.csv")
print("Shape:", df.shape)
df.head()

## 3. Basic inspection

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df["churn_risk_score"].value_counts())
display(df["churn_risk_score"].value_counts(normalize=True))

## 4. Inspect object columns and hidden dirty values

In [ ]:
object_cols = df.select_dtypes(include="object").columns

for col in object_cols:
    print("\n" + "="*70)
    print(f"Column: {col}")
    print(df[col].value_counts(dropna=False).head(15))

## 5. Create a working copy and clean obvious invalid placeholders

In [ ]:
data = df.copy()

# Replace dirty placeholders with proper missing values
data["joined_through_referral"] = data["joined_through_referral"].replace("?", np.nan)
data["medium_of_operation"] = data["medium_of_operation"].replace("?", np.nan)
data["avg_frequency_login_days"] = data["avg_frequency_login_days"].replace("Error", np.nan)

## 6. Convert data types

In [ ]:
data["joining_date"] = pd.to_datetime(data["joining_date"], errors="coerce")
data["avg_frequency_login_days"] = pd.to_numeric(data["avg_frequency_login_days"], errors="coerce")

data.dtypes

## 7. Drop pure identifier columns

In [ ]:
# 'Unnamed: 0' looks like a saved index column
# 'security_no' behaves like a unique ID and is not a meaningful predictive feature
data.drop(columns=["Unnamed: 0", "security_no"], inplace=True)

## 8. Feature engineering

In [ ]:
# 8.1 Referral feature engineering
data["has_referral_id"] = np.where(
    data["referral_id"].isna() | (data["referral_id"] == "xxxxxxxx"),
    0,
    1
)
data.drop(columns=["referral_id"], inplace=True)

# 8.2 Joining date features
data["joining_year"] = data["joining_date"].dt.year
data["joining_month"] = data["joining_date"].dt.month
data["joining_day"] = data["joining_date"].dt.day
data["joining_dayofweek"] = data["joining_date"].dt.dayofweek

reference_date = data["joining_date"].max()
data["customer_tenure_days"] = (reference_date - data["joining_date"]).dt.days

# 8.3 Last visit time features
last_visit_dt = pd.to_datetime(data["last_visit_time"], format="%H:%M:%S", errors="coerce")
data["last_visit_hour"] = last_visit_dt.dt.hour

def map_visit_period(hour):
    if pd.isna(hour):
        return np.nan
    hour = int(hour)
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"

data["visit_period"] = data["last_visit_hour"].apply(map_visit_period)

# Drop original date/time columns after extracting useful information
data.drop(columns=["joining_date", "last_visit_time"], inplace=True)

## 9. Handle suspicious values

In [ ]:
# 'days_since_last_login' should usually be non-negative.
# We treat negative values as invalid and let the imputer handle them later.
neg_count = (data["days_since_last_login"] < 0).sum()
print("Negative values in days_since_last_login:", neg_count)

data.loc[data["days_since_last_login"] < 0, "days_since_last_login"] = np.nan

## 10. Missing values after cleaning

In [ ]:
missing_summary = data.isna().sum().sort_values(ascending=False)
display(missing_summary[missing_summary > 0])

## 11. Define target and feature groups

In [ ]:
target_col = "churn_risk_score"
X = data.drop(columns=[target_col])
y = data[target_col]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

## 12. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=y)
plt.title("Target Distribution: churn_risk_score")
plt.show()

display(X[numeric_features].describe().T)

## 13. Numeric feature distributions

In [ ]:
X[numeric_features].hist(figsize=(18, 12), bins=30)
plt.suptitle("Numeric Feature Distributions", y=1.02)
plt.tight_layout()
plt.show()

## 14. Boxplots for outlier inspection

In [ ]:
for col in numeric_features:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=X[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

## 15. Categorical distributions

In [ ]:
for col in categorical_features:
    plt.figure(figsize=(10, 4))
    order = X[col].value_counts(dropna=False).index
    sns.countplot(data=data, x=col, order=order)
    plt.title(f"Countplot of {col}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 16. Churn rate by categorical feature

In [ ]:
for col in categorical_features:
    churn_rate = data.groupby(col)[target_col].mean().sort_values(ascending=False)
    plt.figure(figsize=(10, 4))
    sns.barplot(x=churn_rate.index, y=churn_rate.values)
    plt.title(f"Churn Rate by {col}")
    plt.ylabel("Mean churn_risk_score")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 17. Churn vs numeric features

In [ ]:
for col in numeric_features:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=data, x=target_col, y=col)
    plt.title(f"{col} vs {target_col}")
    plt.show()

## 18. Leakage and feature-selection notes

In [ ]:
print("""Dropped / transformed columns:
- Unnamed: 0 -> saved index artifact
- security_no -> identifier
- referral_id -> replaced with has_referral_id

Leakage watchlist:
- complaint_status
- feedback
- past_complaint

These are kept for this project under the assumption that they are known at prediction time.
""")

## 19. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

## 20. Build preprocessing pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 21. Baseline model: Logistic Regression

In [ ]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)
baseline_probs = baseline_model.predict_proba(X_test)[:, 1]

print("Baseline Logistic Regression")
print("Accuracy :", accuracy_score(y_test, baseline_preds))
print("Precision:", precision_score(y_test, baseline_preds))
print("Recall   :", recall_score(y_test, baseline_preds))
print("F1       :", f1_score(y_test, baseline_preds))
print("ROC-AUC  :", roc_auc_score(y_test, baseline_probs))

## 22. Helper function for consistent evaluation

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name="Model"):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_prob)
    else:
        y_prob = None
        roc_auc = np.nan

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc
    }, y_pred, y_prob

## 23. Train and compare multiple models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

model_pipelines = {
    name: Pipeline(steps=[("preprocessor", preprocessor), ("model", clf)])
    for name, clf in models.items()
}

results_list = []
for name, pipeline in model_pipelines.items():
    results, _, _ = evaluate_model(pipeline, X_train, y_train, X_test, y_test, model_name=name)
    results_list.append(results)

results_df = pd.DataFrame(results_list).sort_values(by="F1", ascending=False)
display(results_df)

## 24. Cross-validation comparison

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
for name, pipeline in model_pipelines.items():
    f1_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
    cv_results.append({
        "Model": name,
        "CV_F1_Mean": f1_scores.mean(),
        "CV_F1_STD": f1_scores.std()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(by="CV_F1_Mean", ascending=False)
display(cv_results_df)

## 25. Hyperparameter tuning: Random Forest

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2, 4],
    "model__class_weight": [None, "balanced"]
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)
print("Best RF Params:", rf_grid.best_params_)
print("Best RF CV F1 :", rf_grid.best_score_)
best_rf = rf_grid.best_estimator_

## 26. Hyperparameter tuning: Extra Trees

In [ ]:
et_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", ExtraTreesClassifier(random_state=42, n_jobs=-1))
])

et_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2, 4],
    "model__class_weight": [None, "balanced"]
}

et_grid = GridSearchCV(
    estimator=et_pipeline,
    param_grid=et_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

et_grid.fit(X_train, y_train)
print("Best ET Params:", et_grid.best_params_)
print("Best ET CV F1 :", et_grid.best_score_)
best_et = et_grid.best_estimator_

## 27. Compare tuned models on the test set

In [ ]:
def get_test_metrics(model, X_test, y_test, name):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, prob)
    }

tuned_results = [
    get_test_metrics(best_rf, X_test, y_test, "Tuned Random Forest"),
    get_test_metrics(best_et, X_test, y_test, "Tuned Extra Trees")
]

final_results = pd.concat([results_df, pd.DataFrame(tuned_results)], ignore_index=True)
final_results = final_results.sort_values(by="F1", ascending=False)
display(final_results)

## 28. Choose final model

In [ ]:
# Change this if Tuned Extra Trees performs better in your run
final_model = best_rf

## 29. Final evaluation: confusion matrix and classification report

In [ ]:
final_pred = final_model.predict(X_test)
final_prob = final_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, final_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Final Model")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(classification_report(y_test, final_pred))
print("ROC-AUC:", roc_auc_score(y_test, final_prob))

## 30. ROC curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, final_prob)
auc_score = roc_auc_score(y_test, final_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Final Model")
plt.legend()
plt.show()

## 31. Feature importance

In [ ]:
# Fit preprocessor separately to recover transformed feature names
preprocessor.fit(X_train)

cat_ohe = preprocessor.named_transformers_["cat"]["onehot"]
cat_feature_names = cat_ohe.get_feature_names_out(categorical_features)
all_feature_names = list(numeric_features) + list(cat_feature_names)

trained_tree_model = final_model.named_steps["model"]

feature_importances = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": trained_tree_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

display(feature_importances.head(20))

## 32. Plot top important features

In [ ]:
top_n = 20
top_features = feature_importances.head(top_n)

plt.figure(figsize=(10, 8))
sns.barplot(data=top_features, x="Importance", y="Feature")
plt.title(f"Top {top_n} Feature Importances")
plt.tight_layout()
plt.show()

## 33. Save the final model pipeline

In [ ]:
joblib.dump(final_model, "customer_churn_pipeline.pkl")
print("Saved as customer_churn_pipeline.pkl")

## 34. Load model and run a sample prediction

In [ ]:
loaded_model = joblib.load("customer_churn_pipeline.pkl")

sample_input = X_test.iloc[[0]].copy()
sample_pred = loaded_model.predict(sample_input)[0]
sample_prob = loaded_model.predict_proba(sample_input)[0, 1]

print("Predicted class:", sample_pred)
print("Predicted churn probability:", sample_prob)

## 35. Final conclusion template

In [ ]:
print("""Project wrap-up:
1. The churn problem was framed as binary classification.
2. The dataset required real cleaning:
   - missing values
   - placeholder symbols like '?'
   - 'Error' strings in numeric columns
   - identifier columns that should not be modeled directly
3. We engineeredfeatures from joining date, last visit time, and referral data.
4. We built a full sklearn pipeline with imputation, encoding, scaling, and multiple models.
5. We compared models using Accuracy, Precision, Recall, F1, and ROC-AUC.
6. We tuned top ensemble models and selected the best final model based on F1 / ROC-AUC / business tradeoffs.
7. The final model was saved for deployment in a Streamlit or API-based application.
""")